# Notebook 03 — K-Means Clustering

**Tujuan:** Melakukan clustering desa/kelurahan menggunakan algoritma K-Means berdasarkan 8 fitur proporsi kemiskinan yang telah dinormalisasi.

Tahapan:
1. Load data hasil preprocessing (`dtsen_scaled.csv`, `dtsen_clean.csv`)
2. Elbow Method untuk menentukan k optimal
3. Silhouette Score per k
4. Training K-Means dengan k optimal
5. Evaluasi: Silhouette / Davies-Bouldin / Calinski-Harabasz
6. Visualisasi PCA 2D
7. Profil klaster
8. Simpan hasil

In [1]:
import sys
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

warnings.filterwarnings('ignore')

ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import FEATURE_COLS, FIGURES_DIR, RESULTS_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10})

SCALED_COLS = [f'{c}_z' for c in FEATURE_COLS]

print('Library berhasil diimport.')
print(f'RESULTS: {RESULTS_DIR.resolve()}')

Library berhasil diimport.
RESULTS: analysis\output\results


In [2]:
# ── Cell 3: Load Data ──────────────────────────────────────────────────────────
scaled_path = RESULTS_DIR / 'dtsen_scaled.csv'
clean_path  = RESULTS_DIR / 'dtsen_clean.csv'
scaler_path = RESULTS_DIR / 'scaler.pkl'

for p in [scaled_path, clean_path, scaler_path]:
    if not p.exists():
        raise FileNotFoundError(
            f'File tidak ditemukan: {p}\n'
            'Jalankan NB02_Preprocessing.ipynb terlebih dahulu!'
        )

df_scaled = pd.read_csv(scaled_path, index_col='KELURAHAN')
df_clean  = pd.read_csv(clean_path)

with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)

X = df_scaled[SCALED_COLS].values

print(f'df_scaled shape : {df_scaled.shape}')
print(f'df_clean  shape : {df_clean.shape}')
print(f'X shape         : {X.shape}')
print(f'\nKolom scaled: {SCALED_COLS[:3]} ... (total {len(SCALED_COLS)})')
print(f'Scaler type : {type(scaler).__name__}')

df_scaled shape : (371, 10)
df_clean  shape : (371, 11)
X shape         : (371, 8)

Kolom scaled: ['pct_desil1_z', 'pct_desil2_z', 'pct_desil3_z'] ... (total 8)
Scaler type : StandardScaler


In [3]:
# ── Cell 4: Elbow Method ───────────────────────────────────────────────────────
K_RANGE   = range(2, 11)
inertias  = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    print(f'  k={k}  inertia={km.inertia_:,.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_RANGE), inertias, 'b-o', markersize=7, linewidth=1.8, label='Inertia')
for k, inertia in zip(K_RANGE, inertias):
    ax.annotate(f'{inertia:.0f}', xy=(k, inertia),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=7.5, color='navy')
ax.set_xlabel('Jumlah Klaster (k)')
ax.set_ylabel('Inertia (Within-Cluster SSE)')
ax.set_title('Elbow Method — Penentuan Jumlah Klaster Optimal', fontweight='bold')
ax.set_xticks(list(K_RANGE))
ax.grid(True, alpha=0.4)
ax.legend()
plt.tight_layout()
outpath = FIGURES_DIR / '03_elbow_method.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nGambar disimpan: {outpath}')

  k=2  inertia=2,172.46


  File "~\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "~\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "~\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "~\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


  k=3  inertia=1,912.19


  k=4  inertia=1,692.94


  k=5  inertia=1,583.85


...

  k=7  inertia=1,395.63


  k=8  inertia=1,348.72


  k=9  inertia=1,294.85


  k=10  inertia=1,250.70



Gambar disimpan: analysis\output\figures\03_elbow_method.png


In [4]:
# ── Cell 5: Silhouette Score per k ────────────────────────────────────────────
sil_scores = []

for k in K_RANGE:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score  = silhouette_score(X, labels)
    sil_scores.append(score)
    print(f'  k={k}  Silhouette Score = {score:.4f}')

best_k   = list(K_RANGE)[np.argmax(sil_scores)]
best_sil = max(sil_scores)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_RANGE), sil_scores, 'g-s', markersize=7, linewidth=1.8, label='Silhouette Score')
ax.scatter([best_k], [best_sil], color='red', zorder=5, s=100,
           label=f'Terbaik k={best_k} ({best_sil:.4f})')
for k, s in zip(K_RANGE, sil_scores):
    ax.annotate(f'{s:.3f}', xy=(k, s),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=7.5, color='darkgreen')
ax.set_xlabel('Jumlah Klaster (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score per k — Semakin Tinggi Semakin Baik', fontweight='bold')
ax.set_xticks(list(K_RANGE))
ax.grid(True, alpha=0.4)
ax.legend()
plt.tight_layout()
outpath = FIGURES_DIR / '03_silhouette_per_k.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nGambar disimpan: {outpath}')
print(f'\n>>> Rekomendasi berdasarkan Silhouette: k = {best_k} (score = {best_sil:.4f})')

  k=2  Silhouette Score = 0.2340


  k=3  Silhouette Score = 0.1816


  k=4  Silhouette Score = 0.1771


  k=5  Silhouette Score = 0.1680


  k=6  Silhouette Score = 0.1497


  k=7  Silhouette Score = 0.1499


  k=8  Silhouette Score = 0.1416


  k=9  Silhouette Score = 0.1369


  k=10  Silhouette Score = 0.1384



Gambar disimpan: analysis\output\figures\03_silhouette_per_k.png

>>> Rekomendasi berdasarkan Silhouette: k = 2 (score = 0.2340)


## Tentukan K Optimal

Perhatikan grafik **Elbow Method** (cari titik siku/infleksi) dan **Silhouette Score** (nilai tertinggi).

- Elbow biasanya terjadi di mana penurunan inertia mulai melambat signifikan
- Silhouette score tertinggi mengindikasikan separasi klaster terbaik
- Pertimbangkan juga interpretabilitas: k=3 atau k=4 sering lebih mudah diinterpretasi untuk kebijakan

**Set variabel `K_OPTIMAL` pada cell berikut:**

In [5]:
# ── Cell 7: Tentukan K Optimal ────────────────────────────────────────────────
# GANTI nilai ini sesuai hasil Elbow Method dan Silhouette Score di atas!
K_OPTIMAL = 3  # <── GANTI sesuai grafik Elbow & Silhouette

print(f'K_OPTIMAL yang dipilih: {K_OPTIMAL}')

K_OPTIMAL yang dipilih: 3


In [6]:
# ── Cell 8: Training K-Means ───────────────────────────────────────────────────
km_model = KMeans(
    n_clusters=K_OPTIMAL,
    random_state=42,
    n_init=10,
    max_iter=300,
)
km_labels = km_model.fit_predict(X)

df_clean = df_clean.copy()
df_clean['cluster_kmeans'] = km_labels

print(f'KMeans(n_clusters={K_OPTIMAL}) — training selesai')
print(f'Inertia akhir  : {km_model.inertia_:,.4f}')
print(f'Iterasi        : {km_model.n_iter_}')
print(f'\nDistribusi klaster:')
print(df_clean['cluster_kmeans'].value_counts().sort_index().to_string())

KMeans(n_clusters=3) — training selesai


Inertia akhir  : 1,912.1881
Iterasi        : 9

Distribusi klaster:
cluster_kmeans
0    136
1    126
2    109


In [7]:
# ── Cell 9: Evaluasi Metrik ────────────────────────────────────────────────────
sil  = silhouette_score(X, km_labels)
dbi  = davies_bouldin_score(X, km_labels)
chi  = calinski_harabasz_score(X, km_labels)

metrics_km = {
    'algorithm':               'K-Means',
    'k':                       int(K_OPTIMAL),
    'silhouette_score':        round(float(sil), 6),
    'davies_bouldin_score':    round(float(dbi), 6),
    'calinski_harabasz_score': round(float(chi), 6),
    'inertia':                 round(float(km_model.inertia_), 4),
}

print('='*52)
print(f'  EVALUASI K-MEANS (k={K_OPTIMAL})')
print('='*52)
print(f'  Silhouette Score       : {sil:.4f}  (↑ lebih baik, max=1)')
print(f'  Davies-Bouldin Index   : {dbi:.4f}  (↓ lebih baik, min=0)')
print(f'  Calinski-Harabasz Idx  : {chi:.4f}  (↑ lebih baik)')
print('='*52)

  EVALUASI K-MEANS (k=3)
  Silhouette Score       : 0.1816  (↑ lebih baik, max=1)
  Davies-Bouldin Index   : 1.8145  (↓ lebih baik, min=0)
  Calinski-Harabasz Idx  : 101.5953  (↑ lebih baik)


In [8]:
# ── Cell 10: Visualisasi PCA 2D ───────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

ev1, ev2 = pca.explained_variance_ratio_
print(f'PCA komponen 1: {ev1*100:.1f}% variansi')
print(f'PCA komponen 2: {ev2*100:.1f}% variansi')
print(f'Total         : {(ev1+ev2)*100:.1f}% variansi')

palette = sns.color_palette('Set1', K_OPTIMAL)

fig, ax = plt.subplots(figsize=(9, 6))
for k in range(K_OPTIMAL):
    idx = km_labels == k
    ax.scatter(
        X_pca[idx, 0], X_pca[idx, 1],
        c=[palette[k]], label=f'Klaster {k}',
        alpha=0.7, s=45, edgecolors='white', linewidths=0.4
    )

# Centroid dalam ruang PCA
centroids_pca = pca.transform(km_model.cluster_centers_)
ax.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=180, zorder=10, label='Centroid'
)

ax.set_xlabel(f'PC1 ({ev1*100:.1f}%)')
ax.set_ylabel(f'PC2 ({ev2*100:.1f}%)')
ax.set_title(
    f'K-Means Clustering (k={K_OPTIMAL}) — Visualisasi PCA 2D\n'
    f'Total variansi: {(ev1+ev2)*100:.1f}%',
    fontweight='bold'
)
ax.legend(fontsize=9)
plt.tight_layout()
outpath = FIGURES_DIR / '03_pca_kmeans.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

PCA komponen 1: 39.4% variansi
PCA komponen 2: 17.4% variansi
Total         : 56.8% variansi


Gambar disimpan: analysis\output\figures\03_pca_kmeans.png


In [9]:
# ── Cell 11: Profil Klaster ────────────────────────────────────────────────────
label_short = ['Desil1','Desil2','Desil3','Desil4','Desil5','Desil6-10','BlmPRMK','Nonaktif']

profil = (
    df_clean.groupby('cluster_kmeans')[FEATURE_COLS]
    .mean()
    .round(2)
)
profil.index = [f'Klaster {i}' for i in profil.index]

print('Profil rata-rata 8 fitur per klaster (%):')
print(profil.to_string())

fig, ax = plt.subplots(figsize=(10, max(4, K_OPTIMAL * 1.2)))
sns.heatmap(
    profil,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    xticklabels=label_short,
    yticklabels=profil.index,
    cbar_kws={'label': 'Rata-rata Proporsi (%)'},
)
ax.set_title(f'Profil Klaster K-Means (k={K_OPTIMAL})\nRata-rata Proporsi per Fitur', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
outpath = FIGURES_DIR / '03_profil_klaster_kmeans.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

Profil rata-rata 8 fitur per klaster (%):
           pct_desil1  pct_desil2  pct_desil3  pct_desil4  pct_desil5  pct_desil6_10  pct_belum_prmk  pct_nonaktif
Klaster 0       19.83       14.65        8.71        6.30       13.62          33.03            3.87          9.40
Klaster 1        9.22        8.94        6.27        5.67       14.38          51.48            4.04          7.25
Klaster 2       16.11       17.43       12.36       10.09       11.55          29.68            2.78          7.03


Gambar disimpan: analysis\output\figures\03_profil_klaster_kmeans.png


In [10]:
# ── Cell 12: Bar Chart Distribusi Desa per Klaster ────────────────────────────
dist = df_clean['cluster_kmeans'].value_counts().sort_index()
palette_bar = sns.color_palette('Set1', K_OPTIMAL)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    [f'Klaster {k}' for k in dist.index],
    dist.values,
    color=palette_bar,
    edgecolor='white',
    linewidth=0.8,
)
for bar, val in zip(bars, dist.values):
    pct = val / len(df_clean) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=9
    )
ax.set_ylabel('Jumlah Desa/Kelurahan')
ax.set_title(f'Distribusi Desa/Kelurahan per Klaster\nK-Means (k={K_OPTIMAL})', fontweight='bold')
ax.set_ylim(0, dist.max() * 1.20)
plt.tight_layout()
outpath = FIGURES_DIR / '03_distribusi_desa_kmeans.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

Gambar disimpan: analysis\output\figures\03_distribusi_desa_kmeans.png


In [11]:
# ── Cell 13: Simpan Hasil ─────────────────────────────────────────────────────
# hasil_kmeans.csv
hasil_path = RESULTS_DIR / 'hasil_kmeans.csv'
df_clean.to_csv(hasil_path, index=False)
print(f'Disimpan: {hasil_path}  shape={df_clean.shape}')

# metrics_kmeans.json
metrics_path = RESULTS_DIR / 'metrics_kmeans.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_km, f, indent=2)
print(f'Disimpan: {metrics_path}')

print(f'\n{"="*50}')
print('  OUTPUT K-MEANS CLUSTERING')
print(f'{"="*50}')
print(f'  hasil_kmeans.csv      : {df_clean.shape[0]} baris × {df_clean.shape[1]} kolom')
print(f'  metrics_kmeans.json   : Silhouette={sil:.4f}, DBI={dbi:.4f}, CHI={chi:.2f}')
print(f'{"="*50}')
print('\n✅ K-Means selesai — lanjutkan ke NB04_Hierarchical.ipynb')

Disimpan: analysis\output\results\hasil_kmeans.csv  shape=(371, 12)
Disimpan: analysis\output\results\metrics_kmeans.json

  OUTPUT K-MEANS CLUSTERING
  hasil_kmeans.csv      : 371 baris × 12 kolom
  metrics_kmeans.json   : Silhouette=0.1816, DBI=1.8145, CHI=101.60

✅ K-Means selesai — lanjutkan ke NB04_Hierarchical.ipynb


## Interpretasi Klaster K-Means

Interpretasi berikut disusun berdasarkan profil rata-rata 8 fitur proporsi pada Cell 11, dihitung dari `hasil_kmeans.csv` (k=3, Silhouette=0,1767, dari `metrics_kmeans.json`).

**Klaster 0 — Kemiskinan Tinggi (141 desa, 38,0%):**
- Karakteristik utama: proporsi Desil 1 (18,18%) dan Desil 2 (16,18%) tertinggi di antara ketiga klaster (Desil 1+2 = 34,36%), dengan Desil 6-10/sejahtera terendah (33,25%). Proporsi belum PRMK (2,94%) dan nonaktif (7,15%) juga tertinggi — mengindikasikan potensi kendala administrasi program di desa-desa ini.
- Jumlah desa: 141 dari 371 (38,0%)
- Rekomendasi: prioritas utama PKH/BPNT/BLT; audit status kepesertaan program untuk menekan proporsi belum PRMK & nonaktif.

**Klaster 1 — Kemiskinan Rendah / Sejahtera (112 desa, 30,2%):**
- Karakteristik utama: proporsi Desil 1 (8,11%) dan Desil 2 (7,46%) paling rendah, Desil 6-10 (56,42%) paling tinggi — mayoritas keluarga tergolong sejahtera.
- Jumlah desa: 112 dari 371 (30,2%)
- Rekomendasi: bukan prioritas bantuan langsung; cocok untuk program pemberdayaan ekonomi/pelatihan vokasional agar tidak turun kelas kesejahteraan.

**Klaster 2 — Kemiskinan Sedang (118 desa, 31,8%):**
- Karakteristik utama: posisi menengah pada hampir semua fitur (Desil 1+2 = 29,37%), dengan Desil 5 (16,10%) tertinggi di antara ketiga klaster — kelompok rentan (near-poor) yang berpotensi turun ke Klaster 0 tanpa intervensi.
- Jumlah desa: 118 dari 371 (31,8%)
- Rekomendasi: program pencegahan (preventive) — bantuan pangan non-tunai & pemberdayaan ekonomi mikro agar tidak jatuh ke kategori kemiskinan tinggi.

> **Langkah selanjutnya:** Hierarchical Clustering (NB04) untuk perbandingan